In [1]:
import os
import math
import time
from datetime import timedelta
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, TensorDataset
from torchvision import transforms
from sklearn.cluster import MiniBatchKMeans
from sklearn.model_selection import KFold
from catboost import CatBoostRegressor

# -------------------------------------------------------------
# 1. Dataset Resolution & GeoComp 65k Integration
# -------------------------------------------------------------
print("[1/6] Discovering Competition & GeoComp 65k Datasets...")
root_search = [Path("/kaggle/input"), Path(".")]
ground_truth_matches = []
for root in root_search:
    if root.exists():
        ground_truth_matches.extend(list(root.rglob("ground_truth_coordinates.csv")))
if not ground_truth_matches:
    raise FileNotFoundError("Could not find 'ground_truth_coordinates.csv'.")

CSV_PATH = ground_truth_matches[0]
DATA_DIR = CSV_PATH.parent
SAMPLE_SUB_PATH = DATA_DIR / "sample_submission.csv"
GEOJSON_MATCHES = list(DATA_DIR.rglob("*.geojson")) + list(Path("/kaggle/input").rglob("*.geojson"))

train_img_matches = list(DATA_DIR.rglob("images")) + list(Path("/kaggle/input").rglob("images"))
test_img_matches = list(DATA_DIR.rglob("test_images_sampled")) + list(Path("/kaggle/input").rglob("test_images_sampled"))
TRAIN_IMG_DIR = train_img_matches[0] if train_img_matches else DATA_DIR / "training_dataset/noised_dataset/images"
TEST_IMG_DIR = test_img_matches[0] if test_img_matches else DATA_DIR / "test_images_sampled"

df_comp = pd.read_csv(CSV_PATH)
id_col = [c for c in df_comp.columns if any(k in c.lower() for k in ["id", "image", "name"])][0]
lat_col = [c for c in df_comp.columns if "lat" in c.lower()][0]
lon_col = [c for c in df_comp.columns if any(k in c.lower() for k in ["lon", "lng"])][0]

df_comp["file_path"] = df_comp[id_col].apply(
    lambda x: os.path.join(str(TRAIN_IMG_DIR), x if str(x).endswith(".jpg") else f"{x}.jpg")
)
df_comp["is_external"] = False

# Search for external 65k dataset
ext_csv_candidates = [
    p for p in Path("/kaggle/input").rglob("*.csv")
    if p != CSV_PATH and p != SAMPLE_SUB_PATH and "sample" not in p.name.lower()
]
df_ext_list = []
if ext_csv_candidates:
    try:
        df_ext_raw = pd.read_csv(ext_csv_candidates[0])
        e_id = [c for c in df_ext_raw.columns if any(k in c.lower() for k in ["id", "image", "file", "name"])][0]
        e_lat = [c for c in df_ext_raw.columns if "lat" in c.lower()][0]
        e_lon = [c for c in df_ext_raw.columns if any(k in c.lower() for k in ["lon", "lng"])][0]

        df_ext_clean = df_ext_raw.dropna(subset=[e_id, e_lat, e_lon]).copy()
        df_ext_clean = df_ext_clean[
            (df_ext_clean[e_lat].between(-90.0, 90.0)) & (df_ext_clean[e_lon].between(-180.0, 180.0))
        ]

        ext_img_dirs = list(ext_csv_candidates[0].parent.rglob("*.jpg"))
        if ext_img_dirs:
            ext_dir_map = {f.name: str(f) for f in ext_img_dirs[:70000]}
            df_ext_clean["file_path"] = df_ext_clean[e_id].apply(
                lambda x: ext_dir_map.get(str(x) if str(x).endswith(".jpg") else f"{x}.jpg", None)
            )
            df_ext_clean = df_ext_clean.dropna(subset=["file_path"])

        df_ext_clean = df_ext_clean.rename(columns={e_id: id_col, e_lat: lat_col, e_lon: lon_col})
        df_ext_clean["is_external"] = True
        df_ext_65k = df_ext_clean.sample(n=min(65000, len(df_ext_clean)), random_state=42)
        df_ext_list.append(df_ext_65k[[id_col, lat_col, lon_col, "file_path", "is_external"]])
        print(f"[✓] Successfully merged {len(df_ext_65k):,} external 65k GeoComp samples.")
    except Exception as e:
        print(f"[!] External dataset skipped: {e}")

df_all_train = pd.concat([df_comp[[id_col, lat_col, lon_col, "file_path", "is_external"]]] + df_ext_list, ignore_index=True)

if SAMPLE_SUB_PATH.exists():
    df_test = pd.read_csv(SAMPLE_SUB_PATH)
    test_id_col = [c for c in df_test.columns if any(k in c.lower() for k in ["id", "image", "name"])][0]
    df_test["file_path"] = df_test[test_id_col].apply(
        lambda x: os.path.join(str(TEST_IMG_DIR), x if str(x).endswith(".jpg") else f"{x}.jpg")
    )
else:
    test_files = sorted(list(TEST_IMG_DIR.glob("*.jpg")))
    test_id_col = id_col
    df_test = pd.DataFrame({
        id_col: [f.name for f in test_files],
        "file_path": [str(f) for f in test_files]
    })

# -------------------------------------------------------------
# 2. 3D Coordinates & Multi-Granularity Spatial Targets
# -------------------------------------------------------------
lat_rad = np.radians(df_all_train[lat_col].values.astype(float))
lon_rad = np.radians(df_all_train[lon_col].values.astype(float))
df_all_train["x"] = np.cos(lat_rad) * np.cos(lon_rad)
df_all_train["y"] = np.cos(lat_rad) * np.sin(lon_rad)
df_all_train["z"] = np.sin(lat_rad)

K_GRID = 512
kmeans_grid = MiniBatchKMeans(n_clusters=K_GRID, random_state=42, batch_size=4096, max_iter=100)
df_all_train["grid_label"] = kmeans_grid.fit_predict(df_all_train[["x", "y", "z"]].values)
centroids_grid = kmeans_grid.cluster_centers_
centroids_grid = centroids_grid / np.linalg.norm(centroids_grid, axis=1, keepdims=True)
centroid_grid_lat = np.degrees(np.arcsin(np.clip(centroids_grid[:, 2], -1.0, 1.0)))
centroid_grid_lon = np.degrees(np.arctan2(centroids_grid[:, 1], centroids_grid[:, 0]))

def create_semantic_geocells(df, max_total_cells=768):
    try:
        import geopandas as gpd
        from shapely.geometry import Point
        if GEOJSON_MATCHES:
            gdf_borders = gpd.read_file(GEOJSON_MATCHES[0])
            geometry = [Point(xy) for xy in zip(df[lon_col], df[lat_col])]
            gdf_points = gpd.GeoDataFrame(df[[lat_col, lon_col]], geometry=geometry, crs=gdf_borders.crs)
            joined = gpd.sjoin(gdf_points, gdf_borders, how="left", predicate="within")
            country_col = [c for c in joined.columns if any(k in c.lower() for k in ["iso", "name", "country"])][0]
            df["country_id"] = joined[country_col].fillna("UNKNOWN")
        else:
            df["country_id"] = "GLOBAL"
    except Exception:
        df["country_id"] = "GLOBAL"

    semantic_labels = np.zeros(len(df), dtype=int)
    current_label = 0
    semantic_centroids = []

    for country, group in df.groupby("country_id"):
        n_pts = len(group)
        if n_pts == 0:
            continue
        n_sub = max(1, min(25, int(np.ceil(n_pts / 60.0))))
        if current_label + n_sub >= max_total_cells:
            n_sub = max(1, max_total_cells - current_label)

        sub_km = MiniBatchKMeans(n_clusters=n_sub, random_state=42, batch_size=2048, max_iter=50)
        sub_preds = sub_km.fit_predict(group[["x", "y", "z"]].values)
        semantic_labels[group.index] = current_label + sub_preds

        c_xyz = sub_km.cluster_centers_
        c_xyz = c_xyz / np.linalg.norm(c_xyz, axis=1, keepdims=True)
        semantic_centroids.append(c_xyz)
        current_label += n_sub
        if current_label >= max_total_cells:
            break

    df["semantic_label"] = semantic_labels
    return current_label, np.vstack(semantic_centroids)

NUM_SEMANTIC_CLASSES, centroids_sem = create_semantic_geocells(df_all_train, max_total_cells=768)
centroid_sem_lat = np.degrees(np.arcsin(np.clip(centroids_sem[:, 2], -1.0, 1.0)))
centroid_sem_lon = np.degrees(np.arctan2(centroids_sem[:, 1], centroids_sem[:, 0]))

def create_s2_partition_cells(df, n_cells):
    km = MiniBatchKMeans(n_clusters=n_cells, random_state=42, batch_size=4096, max_iter=100)
    labels = km.fit_predict(df[["x", "y", "z"]].values)
    centers = km.cluster_centers_
    centers = centers / np.linalg.norm(centers, axis=1, keepdims=True)
    c_lat = np.degrees(np.arcsin(np.clip(centers[:, 2], -1.0, 1.0)))
    c_lon = np.degrees(np.arctan2(centers[:, 1], centers[:, 0]))
    return labels, centers, c_lat, c_lon

NUM_S2_COARSE = 24
NUM_S2_FINE = 96
df_all_train["s2_coarse_label"], centroids_s2_coarse, centroid_s2_c_lat, centroid_s2_c_lon = create_s2_partition_cells(df_all_train, NUM_S2_COARSE)
df_all_train["s2_fine_label"], centroids_s2_fine, centroid_s2_f_lat, centroid_s2_f_lon = create_s2_partition_cells(df_all_train, NUM_S2_FINE)

df_comp_train = df_all_train[~df_all_train["is_external"]].copy()
val_size = int(0.15 * len(df_comp_train))
val_indices = df_comp_train.sample(n=val_size, random_state=42).index
df_val = df_all_train.loc[val_indices].copy().reset_index(drop=True)
df_train_pool = df_all_train.drop(index=val_indices).copy().reset_index(drop=True)

# -------------------------------------------------------------
# 3. SigLIP 2 Offline Image Embedding Extraction
# -------------------------------------------------------------
OUTPUT_DIR = Path("/kaggle/working/siglip2_cache")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
SIGLIP_MODEL_NAME = "google/siglip-so400m-patch14-384"
IMG_SIZE = 384

siglip_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

siglip_transforms_flip = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=1.0),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

class ImageExtractDataset(Dataset):
    def __init__(self, df, transform, transform_tta=None):
        self.df = df
        self.transform = transform
        self.transform_tta = transform_tta

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        path = self.df.iloc[idx]["file_path"]
        try:
            img = Image.open(path).convert("RGB")
        except Exception:
            img = Image.new("RGB", (IMG_SIZE, IMG_SIZE), color=(0, 0, 0))

        img_orig = self.transform(img)
        if self.transform_tta is not None:
            img_flip = self.transform_tta(img)
            return img_orig, img_flip
        return img_orig

def extract_and_cache_siglip2():
    if (OUTPUT_DIR / "train_embeddings.pt").exists() and (OUTPUT_DIR / "metadata.pt").exists():
        print("[✓] Cached SigLIP 2 embeddings found! Skipping extraction.")
        return

    print(f"\n[2/6] Extracting SigLIP 2 Features ({SIGLIP_MODEL_NAME})...")
    from transformers import SiglipVisionModel
    siglip_model = SiglipVisionModel.from_pretrained(SIGLIP_MODEL_NAME).to(DEVICE)
    if torch.cuda.device_count() > 1:
        siglip_model = nn.DataParallel(siglip_model)
    siglip_model.eval()

    def extract_split(df, desc, use_tta=True):
        ds = ImageExtractDataset(df, siglip_transforms, siglip_transforms_flip if use_tta else None)
        loader = DataLoader(ds, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)
        embed_list = []
        with torch.no_grad():
            for batch in tqdm(loader, desc=desc):
                if use_tta:
                    imgs, imgs_flip = batch
                    imgs = imgs.to(DEVICE, non_blocking=True)
                    imgs_flip = imgs_flip.to(DEVICE, non_blocking=True)
                    with torch.amp.autocast("cuda", enabled=(DEVICE.type == "cuda")):
                        out1 = siglip_model(imgs)
                        out2 = siglip_model(imgs_flip)

                        f1 = out1.pooler_output if hasattr(out1, "pooler_output") else out1.last_hidden_state.mean(dim=1)
                        f2 = out2.pooler_output if hasattr(out2, "pooler_output") else out2.last_hidden_state.mean(dim=1)

                        feats = 0.5 * (F.normalize(f1, p=2, dim=-1) + F.normalize(f2, p=2, dim=-1))
                        feats = F.normalize(feats, p=2, dim=-1)
                else:
                    imgs = batch.to(DEVICE, non_blocking=True)
                    with torch.amp.autocast("cuda", enabled=(DEVICE.type == "cuda")):
                        out = siglip_model(imgs)
                        feats = out.pooler_output if hasattr(out, "pooler_output") else out.last_hidden_state.mean(dim=1)
                        feats = F.normalize(feats, p=2, dim=-1)

                embed_list.append(feats.half().cpu())
        return torch.cat(embed_list, dim=0)

    train_emb = extract_split(df_train_pool, "Extracting Train Embeddings", use_tta=True)
    val_emb = extract_split(df_val, "Extracting Val Embeddings", use_tta=True)
    test_emb = extract_split(df_test, "Extracting Test Embeddings", use_tta=True)

    torch.save(train_emb, OUTPUT_DIR / "train_embeddings.pt")
    torch.save(val_emb, OUTPUT_DIR / "val_embeddings.pt")
    torch.save(test_emb, OUTPUT_DIR / "test_embeddings.pt")

    metadata = {
        "train_xyz": torch.tensor(df_train_pool[["x", "y", "z"]].values, dtype=torch.float32),
        "val_xyz": torch.tensor(df_val[["x", "y", "z"]].values, dtype=torch.float32),
        "val_lat": torch.tensor(df_val[lat_col].values.astype(float), dtype=torch.float32),
        "val_lon": torch.tensor(df_val[lon_col].values.astype(float), dtype=torch.float32),
        "test_ids": [str(x) for x in df_test[test_id_col].tolist()]
    }
    torch.save(metadata, OUTPUT_DIR / "metadata.pt")

extract_and_cache_siglip2()

# -------------------------------------------------------------
# 4. Architecture & Adaptive Dynamic Loss
# -------------------------------------------------------------
train_emb = torch.load(OUTPUT_DIR / "train_embeddings.pt", map_location="cpu").float()
val_emb = torch.load(OUTPUT_DIR / "val_embeddings.pt", map_location="cpu").float()
test_emb = torch.load(OUTPUT_DIR / "test_embeddings.pt", map_location="cpu").float()
meta = torch.load(OUTPUT_DIR / "metadata.pt", map_location="cpu")

train_xyz = meta["train_xyz"]
val_xyz = meta["val_xyz"]
val_lat = meta["val_lat"].numpy()
val_lon = meta["val_lon"].numpy()
test_ids = meta["test_ids"]
EMBED_DIM = train_emb.shape[1]

class SigLIP5HeadNetwork(nn.Module):
    def __init__(self, embed_dim, num_grid=512, num_sem=768, num_s2_c=24, num_s2_f=96):
        super().__init__()
        self.trunk = nn.Sequential(
            nn.Linear(embed_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.GELU(),
            nn.Dropout(0.3)
        )
        self.head_reg = nn.Sequential(
            nn.Linear(1024, 256), nn.BatchNorm1d(256), nn.GELU(), nn.Dropout(0.2), nn.Linear(256, 4)
        )
        self.head_grid = nn.Sequential(
            nn.Linear(1024, 512), nn.BatchNorm1d(512), nn.GELU(), nn.Dropout(0.2), nn.Linear(512, num_grid)
        )
        self.head_sem = nn.Sequential(
            nn.Linear(1024, 512), nn.BatchNorm1d(512), nn.GELU(), nn.Dropout(0.2), nn.Linear(512, num_sem)
        )
        self.head_s2_c = nn.Sequential(
            nn.Linear(1024, 256), nn.BatchNorm1d(256), nn.GELU(), nn.Dropout(0.2), nn.Linear(256, num_s2_c)
        )
        self.head_s2_f = nn.Sequential(
            nn.Linear(1024, 256), nn.BatchNorm1d(256), nn.GELU(), nn.Dropout(0.2), nn.Linear(256, num_s2_f)
        )

    def forward(self, feats):
        h = self.trunk(feats)
        reg_out = self.head_reg(h)
        pred_xyz = F.normalize(reg_out[:, :3], p=2, dim=-1)
        log_sigma = reg_out[:, 3:4]

        return pred_xyz, log_sigma, self.head_grid(h), self.head_sem(h), self.head_s2_c(h), self.head_s2_f(h)

class DynamicAdaptiveLoss(nn.Module):
    def __init__(self, c_grid, c_sem, c_s2_c, c_s2_f, tau=175.0):
        super().__init__()
        self.register_buffer("c_grid", torch.tensor(c_grid, dtype=torch.float32))
        self.register_buffer("c_sem", torch.tensor(c_sem, dtype=torch.float32))
        self.register_buffer("c_s2_c", torch.tensor(c_s2_c, dtype=torch.float32))
        self.register_buffer("c_s2_f", torch.tensor(c_s2_f, dtype=torch.float32))
        self.tau = tau
        
        # 5 Unfrozen Learnable Scalar Parameters for loss auto-balancing
        self.log_vars = nn.Parameter(torch.zeros(5, dtype=torch.float32))

    def _smooth_targets(self, true_xyz, centroids):
        cos_sim = torch.matmul(true_xyz, centroids.T).clamp(-1.0 + 1e-7, 1.0 - 1e-7)
        dist_km = torch.acos(cos_sim) * 6371.0
        return F.softmax(-dist_km / self.tau, dim=-1)

    def forward(self, pred_xyz, log_sigma, g_logits, sem_logits, s2c_logits, s2f_logits, true_xyz):
        cos_sim = torch.sum(pred_xyz * true_xyz, dim=-1).clamp(-1.0 + 1e-7, 1.0 - 1e-7)
        dist_km = torch.acos(cos_sim) * 6371.0
        sigma = torch.exp(log_sigma).squeeze(-1) + 1e-4
        l_reg = ((dist_km ** 2) / (2 * (sigma ** 2)) + torch.log(sigma)).mean()

        l_grid = -torch.sum(self._smooth_targets(true_xyz, self.c_grid) * F.log_softmax(g_logits, dim=-1), dim=-1).mean()
        l_sem = -torch.sum(self._smooth_targets(true_xyz, self.c_sem) * F.log_softmax(sem_logits, dim=-1), dim=-1).mean()
        l_s2c = -torch.sum(self._smooth_targets(true_xyz, self.c_s2_c) * F.log_softmax(s2c_logits, dim=-1), dim=-1).mean()
        l_s2f = -torch.sum(self._smooth_targets(true_xyz, self.c_s2_f) * F.log_softmax(s2f_logits, dim=-1), dim=-1).mean()

        # Dynamic weighting via learned variance
        w = torch.exp(-self.log_vars)
        total = (w[0] * l_reg + self.log_vars[0]) + \
                (w[1] * l_grid + self.log_vars[1]) + \
                (w[2] * l_sem + self.log_vars[2]) + \
                (w[3] * l_s2c + self.log_vars[3]) + \
                (w[4] * l_s2f + self.log_vars[4])

        return total, dist_km.mean()

# -------------------------------------------------------------
# 5. Fast 8-Epoch Vector Training Loop
# -------------------------------------------------------------
EPOCHS = 8
BATCH_SIZE = 256
LR = 1e-3

train_ds = TensorDataset(train_emb, train_xyz)
val_ds = TensorDataset(val_emb, val_xyz)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

model = SigLIP5HeadNetwork(EMBED_DIM, K_GRID, NUM_SEMANTIC_CLASSES, NUM_S2_COARSE, NUM_S2_FINE).to(DEVICE)
criterion = DynamicAdaptiveLoss(centroids_grid, centroids_sem, centroids_s2_coarse, centroids_s2_fine, tau=175.0).to(DEVICE)

optimizer = torch.optim.AdamW(
    list(model.parameters()) + list(criterion.parameters()), 
    lr=LR, 
    weight_decay=1e-3
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

print(f"\n[4/6] Training Head Network ({EPOCHS} Epochs, LR={LR})...")
best_val_dist = float("inf")
start_time = time.time()

for epoch in range(EPOCHS):
    model.train()
    criterion.train()
    running_loss = 0.0

    for b_emb, b_xyz in train_loader:
        b_emb, b_xyz = b_emb.to(DEVICE), b_xyz.to(DEVICE)

        optimizer.zero_grad()
        p_xyz, l_sig, g_log, sem_log, s2c_log, s2f_log = model(b_emb)
        loss, _ = criterion(p_xyz, l_sig, g_log, sem_log, s2c_log, s2f_log, b_xyz)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * b_emb.size(0)

    scheduler.step()

    model.eval()
    criterion.eval()
    running_val_dist = 0.0

    with torch.no_grad():
        for b_emb, b_xyz in val_loader:
            b_emb, b_xyz = b_emb.to(DEVICE), b_xyz.to(DEVICE)
            p_xyz, l_sig, g_log, sem_log, s2c_log, s2f_log = model(b_emb)
            _, dist_km = criterion(p_xyz, l_sig, g_log, sem_log, s2c_log, s2f_log, b_xyz)
            running_val_dist += dist_km.item() * b_emb.size(0)

    val_dist = running_val_dist / len(val_ds)
    train_loss = running_loss / len(train_ds)
    print(f"Epoch [{epoch + 1:02d}/{EPOCHS}] | Train Loss: {train_loss:.4f} | Val Dist: {val_dist:.1f} km")

    if val_dist < best_val_dist:
        best_val_dist = val_dist
        torch.save(model.state_dict(), "/kaggle/working/best_siglip2_8epoch.pth")

print(f"[✓] Completed in {timedelta(seconds=int(time.time() - start_time))} | Best Val Dist: {best_val_dist:.1f} km")

# -------------------------------------------------------------
# 6. Feature Stacking & CatBoost Meta-Learner
# -------------------------------------------------------------
print("\n[5/6] Building Stacking Features for CatBoost Meta-Learner...")
model.load_state_dict(torch.load("/kaggle/working/best_siglip2_8epoch.pth", map_location=DEVICE))
model.eval()

TOP_K = 5

def haversine_np(lat1, lon1, lat2, lon2):
    R = 6371.0
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi, dlambda = np.radians(lat2 - lat1), np.radians(lon2 - lon1)
    a = np.sin(dphi / 2)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2)**2
    return 2 * R * np.arcsin(np.clip(np.sqrt(np.clip(a, 0, 1)), 0, 1))

def build_5head_features(embed_tensor):
    feat_chunks = []
    loader = DataLoader(TensorDataset(embed_tensor), batch_size=512, shuffle=False)
    with torch.no_grad():
        for (b_emb,) in loader:
            b_emb = b_emb.to(DEVICE)
            p_xyz, l_sig, g_log, sem_log, s2c_log, s2f_log = model(b_emb)

            xyz = p_xyz.cpu().numpy()
            sigma = torch.exp(l_sig).cpu().numpy()
            p_grid = F.softmax(g_log, dim=-1).cpu().numpy()
            p_sem = F.softmax(sem_log, dim=-1).cpu().numpy()
            p_s2c = F.softmax(s2c_log, dim=-1).cpu().numpy()
            p_s2f = F.softmax(s2f_log, dim=-1).cpu().numpy()

            g_idx = np.argsort(-p_grid, axis=1)[:, :TOP_K]
            g_prob = np.take_along_axis(p_grid, g_idx, axis=1)
            g_lat, g_lon = centroid_grid_lat[g_idx], centroid_grid_lon[g_idx]

            s_idx = np.argsort(-p_sem, axis=1)[:, :TOP_K]
            s_prob = np.take_along_axis(p_sem, s_idx, axis=1)
            s_lat, s_lon = centroid_sem_lat[s_idx], centroid_sem_lon[s_idx]

            s2f_idx = np.argsort(-p_s2f, axis=1)[:, :TOP_K]
            s2f_prob = np.take_along_axis(p_s2f, s2f_idx, axis=1)
            s2f_lat, s2f_lon = centroid_s2_f_lat[s2f_idx], centroid_s2_f_lon[s2f_idx]

            sem_exp = np.einsum("nk,kc->nc", p_sem, centroids_sem)
            sem_exp = sem_exp / (np.linalg.norm(sem_exp, axis=1, keepdims=True) + 1e-8)

            r_lat = np.degrees(np.arcsin(np.clip(xyz[:, 2], -1.0, 1.0)))
            r_lon = np.degrees(np.arctan2(xyz[:, 1], xyz[:, 0]))
            d_reg_sem = haversine_np(r_lat, r_lon, s_lat[:, 0], s_lon[:, 0]).reshape(-1, 1)

            feats = np.concatenate([
                xyz, g_lat, g_lon, g_prob, s_lat, s_lon, s_prob,
                s2f_lat, s2f_lon, s2f_prob, sem_exp, d_reg_sem, p_s2c, sigma
            ], axis=1)
            feat_chunks.append(feats)
    return np.nan_to_num(np.concatenate(feat_chunks, axis=0), nan=0.0)

X_val = build_5head_features(val_emb)
X_test = build_5head_features(test_emb)

catboost_params = {
    "loss_function": "MultiRMSE",
    "iterations": 600,
    "depth": 5,
    "learning_rate": 0.03,
    "l2_leaf_reg": 8.0,
    "bootstrap_type": "Bernoulli",
    "subsample": 0.8,
    "colsample_bylevel": 0.8,
    "verbose": False
}

true_val_xyz = val_xyz.numpy()
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_pred_xyz = np.zeros_like(true_val_xyz)

for fold, (trn_idx, hld_idx) in enumerate(kf.split(X_val)):
    cb = CatBoostRegressor(**catboost_params)
    cb.fit(
        X_val[trn_idx], true_val_xyz[trn_idx],
        eval_set=(X_val[hld_idx], true_val_xyz[hld_idx]),
        early_stopping_rounds=40
    )
    oof_pred_xyz[hld_idx] = cb.predict(X_val[hld_idx])

oof_pred_xyz = oof_pred_xyz / (np.linalg.norm(oof_pred_xyz, axis=1, keepdims=True) + 1e-8)
oof_lat = np.degrees(np.arcsin(np.clip(oof_pred_xyz[:, 2], -1.0, 1.0)))
oof_lon = np.degrees(np.arctan2(oof_pred_xyz[:, 1], oof_pred_xyz[:, 0]))
oof_error_km = haversine_np(oof_lat, oof_lon, val_lat, val_lon)

print(f"[✓] Final SigLIP 2 OOF Mean Error: {oof_error_km.mean():.1f} km")

meta_model = CatBoostRegressor(**catboost_params)
meta_model.fit(X_val, true_val_xyz)

radius_params = {
    "loss_function": "Quantile:alpha=0.80",
    "iterations": 400,
    "depth": 5,
    "learning_rate": 0.03,
    "l2_leaf_reg": 5.0,
    "verbose": False
}
radius_model = CatBoostRegressor(**radius_params)
radius_model.fit(X_val, oof_error_km)

# -------------------------------------------------------------
# 7. Final Submission Generation
# -------------------------------------------------------------
print("\n[6/6] Generating Final Test Predictions...")
pred_test_xyz = meta_model.predict(X_test)
pred_test_xyz = pred_test_xyz / (np.linalg.norm(pred_test_xyz, axis=1, keepdims=True) + 1e-8)

final_lat = np.degrees(np.arcsin(np.clip(pred_test_xyz[:, 2], -1.0, 1.0)))
final_lon = np.degrees(np.arctan2(pred_test_xyz[:, 1], pred_test_xyz[:, 0]))
final_radius = np.clip(radius_model.predict(X_test), 25.0, 3500.0)

df_submission = pd.DataFrame({
    "image_id": test_ids,
    "pred_lat": np.round(final_lat, 6),
    "pred_lon": np.round(final_lon, 6),
    "pred_radius_km": np.round(final_radius, 2)
})

df_submission.to_csv("/kaggle/working/submission_siglip2_8epoch.csv", index=False)
print(f"[✓] 'submission_siglip2_8epoch.csv' generated ({len(df_submission)} rows).")

[1/6] Discovering Competition & GeoComp 65k Datasets...

[2/6] Extracting SigLIP 2 Features (google/siglip-so400m-patch14-384)...


config.json:   0%|          | 0.00/576 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.51G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/448 [00:00<?, ?it/s]

SiglipVisionModel LOAD REPORT from: google/siglip-so400m-patch14-384
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...26}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...26}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...26}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.head.bias                                         | UNEXPECTED |  | 
text_model.encoder.layers.{0...26}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...26}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...26}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...26}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...26}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...26}.self_attn.out_pr

Extracting Train Embeddings:   0%|          | 0/253 [00:00<?, ?it/s]

Extracting Val Embeddings:   0%|          | 0/45 [00:00<?, ?it/s]

Extracting Test Embeddings:   0%|          | 0/8 [00:00<?, ?it/s]


[4/6] Training Head Network (8 Epochs, LR=0.001)...
Epoch [01/8] | Train Loss: 3384019.7073 | Val Dist: 3213.7 km
Epoch [02/8] | Train Loss: 278296.0374 | Val Dist: 2893.9 km
Epoch [03/8] | Train Loss: 173367.8027 | Val Dist: 2844.9 km
Epoch [04/8] | Train Loss: 125257.5161 | Val Dist: 2803.4 km
Epoch [05/8] | Train Loss: 98680.8572 | Val Dist: 2782.6 km
Epoch [06/8] | Train Loss: 86153.0860 | Val Dist: 2774.9 km
Epoch [07/8] | Train Loss: 79063.8583 | Val Dist: 2767.1 km
Epoch [08/8] | Train Loss: 77926.2613 | Val Dist: 2762.4 km
[✓] Completed in 0:00:07 | Best Val Dist: 2762.4 km

[5/6] Building Stacking Features for CatBoost Meta-Learner...
[✓] Final SigLIP 2 OOF Mean Error: 1974.3 km

[6/6] Generating Final Test Predictions...
[✓] 'submission_siglip2_8epoch.csv' generated (500 rows).
